# E-Commerce Customer & Revenue Intelligence Analytics
## Phase 2: Data Cleaning, Profiling & Feature Engineering
**Author:** Vansh Kesarwani  
**Goal:** Entry-Level Data Analyst Portfolio / Internship Project  
**Dataset:** Olist Brazilian E-Commerce Dataset (Kaggle)

---
### Notebook Workflow
1. Environment Setup & Configuration
2. Orders Table: Datetime Parsing, Lead Times & Delivery Performance
3. Order Items Table: Financial Metrics & Freight Ratios
4. Products Table: Portuguese-to-English Translation & Missing Values
5. Payments Table: Anomaly Handling & Installment Segmentation
6. Reviews Table: Sentiment Classification & Comment Auditing
7. Customers & Sellers: Geographic Text Standardization
8. Export Cleaned Staging Datasets & SQLite Database


In [1]:
import os
import sqlite3
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
RAW_DIR = "../dataset"
CLEAN_DIR = "../dataset/cleaned"
os.makedirs(CLEAN_DIR, exist_ok=True)
print("Environment initialized successfully.")

Environment initialized successfully.


### 1. Cleaning Orders: Parsing Timestamps & Calculating Delivery Metrics

In [2]:
df_orders = pd.read_csv(os.path.join(RAW_DIR, "olist_orders_dataset.csv"))

# Convert timestamps
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in timestamp_cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors="coerce")

# Time attributes
df_orders["purchase_year"] = df_orders["order_purchase_timestamp"].dt.year
df_orders["purchase_month"] = df_orders["order_purchase_timestamp"].dt.month
df_orders["purchase_year_month"] = df_orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
df_orders["purchase_day_name"] = df_orders["order_purchase_timestamp"].dt.day_name()
df_orders["purchase_hour"] = df_orders["order_purchase_timestamp"].dt.hour

# Operational durations (in days)
df_orders["delivery_lead_time_days"] = ((df_orders["order_delivered_customer_date"] - df_orders["order_purchase_timestamp"]).dt.total_seconds() / 86400.0).round(2)
df_orders["estimated_lead_time_days"] = ((df_orders["order_estimated_delivery_date"] - df_orders["order_purchase_timestamp"]).dt.total_seconds() / 86400.0).round(2)
df_orders["delivery_delay_days"] = ((df_orders["order_delivered_customer_date"] - df_orders["order_estimated_delivery_date"]).dt.total_seconds() / 86400.0).round(2)

def categorize_delivery(row):
    if row["order_status"] != "delivered":
        return "Not Delivered"
    if pd.isnull(row["order_delivered_customer_date"]):
        return "Delivered (Missing Date)"
    if row["order_delivered_customer_date"] <= row["order_estimated_delivery_date"]:
        return "On-Time"
    return "Late"

df_orders["delivery_performance"] = df_orders.apply(categorize_delivery, axis=1)
df_orders["is_late"] = (df_orders["delivery_performance"] == "Late").astype(int)
print("Delivery Performance Breakdown:")
print(df_orders["delivery_performance"].value_counts())
df_orders.head(3)

Delivery Performance Breakdown:
delivery_performance
On-Time                     88644
Late                         7826
Not Delivered                2963
Delivered (Missing Date)        8
Name: count, dtype: int64


### 2. Cleaning Order Items: Pricing & Freight Unit Economics

In [3]:
df_items = pd.read_csv(os.path.join(RAW_DIR, "olist_order_items_dataset.csv"))
df_items["shipping_limit_date"] = pd.to_datetime(df_items["shipping_limit_date"], errors="coerce")
df_items["total_item_value"] = (df_items["price"] + df_items["freight_value"]).round(2)
df_items["freight_ratio"] = np.where(df_items["price"] > 0, (df_items["freight_value"] / df_items["price"]).round(4), 0.0)
print(f"Total Merchandise Volume: R$ {df_items['price'].sum():,.2f}")
print(f"Total Freight Volume:     R$ {df_items['freight_value'].sum():,.2f}")
df_items.head(3)

Total Merchandise Volume: R$ 13,591,643.70
Total Freight Volume:     R$ 2,251,909.54


### 3. Cleaning Products: Resolving Category Translations & Missing Values

In [4]:
df_prod = pd.read_csv(os.path.join(RAW_DIR, "olist_products_dataset.csv"))
df_trans = pd.read_csv(os.path.join(RAW_DIR, "product_category_name_translation.csv"))

# Custom translations for missing categories discovered in profiling
custom_translations = {
    "portateis_cozinha_e_preparadores_de_alimentos": "kitchen_and_food_preparers",
    "pc_gamer": "pc_gamer"
}
df_prod = df_prod.merge(df_trans, on="product_category_name", how="left")
for pt, en in custom_translations.items():
    df_prod.loc[df_prod["product_category_name"] == pt, "product_category_name_english"] = en

df_prod["product_category_name_english"] = df_prod["product_category_name_english"].fillna("other_unknown")
df_prod["category_clean"] = df_prod["product_category_name_english"].str.replace("_", " ").str.title()

# Fill minor dimension nulls with medians
for col in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm", "product_photos_qty"]:
    df_prod[col] = df_prod[col].fillna(df_prod[col].median())

print(f"Clean distinct product categories: {df_prod['category_clean'].nunique()}")
df_prod.head(3)

Clean distinct product categories: 74


### 4. Cleaning Payments: Filtering Undefined Artifacts & Segmenting Installments

In [5]:
df_pay = pd.read_csv(os.path.join(RAW_DIR, "olist_order_payments_dataset.csv"))
# Drop 3 undefined records with 0 value
df_pay = df_pay[df_pay["payment_type"] != "not_defined"].copy()
df_pay["payment_type_clean"] = df_pay["payment_type"].str.replace("_", " ").str.title()

def installment_bracket(val):
    if val == 1:
        return "1x (Single Payment)"
    elif 2 <= val <= 3:
        return "2x - 3x"
    elif 4 <= val <= 6:
        return "4x - 6x"
    elif 7 <= val <= 10:
        return "7x - 10x"
    return "11x - 24x"

df_pay["installment_bracket"] = df_pay["payment_installments"].apply(installment_bracket)
df_pay["is_installment"] = (df_pay["payment_installments"] > 1).astype(int)
print("Payment Type Distribution:")
print(df_pay["payment_type_clean"].value_counts())
df_pay.head(3)

Payment Type Distribution:
payment_type_clean
Credit Card    76795
Boleto         19784
Voucher         5775
Debit Card      1529
Name: count, dtype: int64


### 5. Cleaning Reviews: Text Handling & Sentiment Grouping

In [6]:
df_rev = pd.read_csv(os.path.join(RAW_DIR, "olist_order_reviews_dataset.csv"))
df_rev["review_creation_date"] = pd.to_datetime(df_rev["review_creation_date"], errors="coerce")
df_rev["review_answer_timestamp"] = pd.to_datetime(df_rev["review_answer_timestamp"], errors="coerce")
df_rev["review_comment_title"] = df_rev["review_comment_title"].fillna("")
df_rev["review_comment_message"] = df_rev["review_comment_message"].fillna("")
df_rev["has_comment"] = (df_rev["review_comment_message"] != "").astype(int)

def review_sentiment(score):
    if score >= 4:
        return "Positive (4-5 Stars)"
    elif score == 3:
        return "Neutral (3 Stars)"
    return "Negative (1-2 Stars)"

df_rev["sentiment_group"] = df_rev["review_score"].apply(review_sentiment)
print("Review Sentiment Distribution:")
print(df_rev["sentiment_group"].value_counts())
df_rev.head(3)

Review Sentiment Distribution:
sentiment_group
Positive (4-5 Stars)    76470
Negative (1-2 Stars)    14575
Neutral (3 Stars)        8179
Name: count, dtype: int64


### 6. Standardizing Customers & Sellers

In [7]:
df_cust = pd.read_csv(os.path.join(RAW_DIR, "olist_customers_dataset.csv"))
df_cust["customer_city"] = df_cust["customer_city"].str.strip().str.title()
df_cust["customer_state"] = df_cust["customer_state"].str.strip().str.upper()

df_sellers = pd.read_csv(os.path.join(RAW_DIR, "olist_sellers_dataset.csv"))
df_sellers["seller_city"] = df_sellers["seller_city"].str.strip().str.title()
df_sellers["seller_state"] = df_sellers["seller_state"].str.strip().str.upper()

print(f"Unique customer_unique_id count: {df_cust['customer_unique_id'].nunique():,}")
print(f"Unique sellers count: {len(df_sellers):,}")

Unique customer_unique_id count: 96,096
Unique sellers count: 3,095


### 7. Export Cleaned Staging Datasets to CSV & Centralized SQLite DB

In [8]:
df_orders.to_csv(os.path.join(CLEAN_DIR, "orders_cleaned.csv"), index=False)
df_items.to_csv(os.path.join(CLEAN_DIR, "order_items_cleaned.csv"), index=False)
df_prod.to_csv(os.path.join(CLEAN_DIR, "products_cleaned.csv"), index=False)
df_pay.to_csv(os.path.join(CLEAN_DIR, "order_payments_cleaned.csv"), index=False)
df_rev.to_csv(os.path.join(CLEAN_DIR, "order_reviews_cleaned.csv"), index=False)
df_cust.to_csv(os.path.join(CLEAN_DIR, "customers_cleaned.csv"), index=False)
df_sellers.to_csv(os.path.join(CLEAN_DIR, "sellers_cleaned.csv"), index=False)

db_path = os.path.join(CLEAN_DIR, "olist_analytics.db")
conn = sqlite3.connect(db_path)
df_orders.to_sql("orders", conn, if_exists="replace", index=False)
df_items.to_sql("order_items", conn, if_exists="replace", index=False)
df_prod.to_sql("products", conn, if_exists="replace", index=False)
df_pay.to_sql("order_payments", conn, if_exists="replace", index=False)
df_rev.to_sql("order_reviews", conn, if_exists="replace", index=False)
df_cust.to_sql("customers", conn, if_exists="replace", index=False)
df_sellers.to_sql("sellers", conn, if_exists="replace", index=False)
conn.close()
print("All clean tables exported to CSV and indexed in SQLite database successfully!")

All clean tables exported to CSV and indexed in SQLite database successfully!
